# Daily Challenge — Build a Tiny Agent with Tools

## Guided student notebook

Build a beginner agent with `smolagents`. The agent must:

- search a small internal knowledge base;
- add or multiply two numbers;
- select and execute tools through `ToolCallingAgent`;
- keep final answers short;
- cite internal evidence with tags such as `[kb:1]`;
- admit when the KB has no evidence.

The notebook uses a deterministic stub by default, so no API key or model
download is required.

## Agent loop

```text
Question
   ↓
Model chooses a tool
   ↓
Trusted Python tool executes
   ↓
Observation returns to the model
   ↓
Model produces final_answer
```

# 0. Install dependencies

In [ ]:
# Install the stable smolagents version used in this exercise.
# The transformers extra is required only for the optional local model.

%pip install -qU \
    "smolagents[transformers]==1.26.0" \
    "wikipedia>=1.4,<2"

In [ ]:
# Imports used in the exercise.

import json
import re
from typing import Any

from smolagents import (
    Model,
    Tool,
    ToolCallingAgent,
    TransformersModel,
)

from smolagents.models import (
    ChatMessage,
    ChatMessageToolCall,
    ChatMessageToolCallFunction,
    MessageRole,
)

# 1. Define the knowledge base

In [ ]:
# TODO:
# Add 5–8 short snippets.
#
# Every item must contain:
# - a stable source tag such as kb:1;
# - a short text passage;
# - useful keywords for matching.

kb_snippets = [
    {
        "source": "kb:1",
        "text": (
            "An agentic AI loop observes a task, plans the next action, "
            "selects a tool when needed, inspects the observation, and "
            "continues until it can answer."
        ),
        "keywords": [
            "agentic",
            "agent",
            "loop",
            "plan",
            "observation",
        ],
    },

    # TODO: Add at least four more source-tagged snippets.
]

print("KB entries:", len(kb_snippets))

# 2. Create the tools

In [ ]:
# Common words are ignored during keyword matching.
KB_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can",
    "do", "does", "for", "from", "how", "i", "in", "is", "it",
    "of", "on", "or", "the", "this", "to", "what", "when",
    "where", "which", "who", "why", "with",
}


def tokenize(text: str) -> set[str]:
    """Return lowercase words that are useful for matching."""

    tokens = re.findall(
        r"[a-z0-9]+(?:-[a-z0-9]+)?",
        text.lower(),
    )

    return {
        token
        for token in tokens
        if token not in KB_STOPWORDS
        and len(token) > 1
    }

## 2.1 KBLookupTool

In [ ]:
class KBLookupTool(Tool):
    """Search the internal list for source-tagged evidence."""

    # TODO: Complete the tool metadata.
    name = "kb_lookup"

    description = (
        "TODO: Explain when the agent should use this tool "
        "and what it returns."
    )

    inputs = {
        "query": {
            "type": "string",
            "description": "TODO: Describe the search query.",
        },
    }

    output_type = "string"

    def __init__(
        self,
        kb: list[dict[str, Any]],
        max_results: int = 3,
    ):
        super().__init__()

        self.kb = kb
        self.max_results = max_results

    def forward(self, query: str) -> str:
        """Return relevant source-tagged snippets."""

        # TODO:
        # 1. Tokenize the query.
        # 2. Score every KB item by keyword overlap.
        # 3. Sort the matches from highest to lowest score.
        # 4. Return up to max_results lines:
        #       [kb:N] passage text
        # 5. When no item matches, return:
        #       "No KB evidence found. Ask a more specific follow-up question."

        raise NotImplementedError

## 2.2 MathTool

In [ ]:
class MathTool(Tool):
    """Add or multiply two numbers."""

    name = "math_tool"

    description = (
        "Add or multiply exactly two numbers. "
        "Set op to either 'add' or 'multiply'."
    )

    # TODO:
    # Declare a, b, and op using smolagents input schemas.
    inputs = {
        # "a": {...},
        # "b": {...},
        # "op": {...},
    }

    output_type = "string"

    def forward(
        self,
        a: float,
        b: float,
        op: str,
    ) -> str:
        """Perform one validated operation."""

        # TODO:
        # - add when op == "add";
        # - multiply when op == "multiply";
        # - raise ValueError for any unsupported operation;
        # - return the result as a string.

        raise NotImplementedError

In [ ]:
# TODO: Instantiate both tools.

# kb_tool = ...
# math_tool = ...

# 3. Create the default model

A `ToolCallingAgent` needs a model that returns structured tool calls.

For this exercise, implement or reuse a deterministic `Model` stub so the
notebook works without an API key.

The optional Transformers branch is included later.

In [ ]:
def make_tool_call(
    name: str,
    arguments: dict[str, Any],
    call_id: str,
) -> ChatMessage:
    """Create one assistant message containing a tool call."""

    return ChatMessage(
        role=MessageRole.ASSISTANT,
        content="",
        tool_calls=[
            ChatMessageToolCall(
                function=ChatMessageToolCallFunction(
                    name=name,
                    arguments=arguments,
                ),
                id=call_id,
                type="function",
            )
        ],
    )

In [ ]:
class DeterministicToolModel(Model):
    """TODO: Implement a no-token model for this exercise."""

    def __init__(self):
        super().__init__(
            model_id="deterministic-tool-stub"
        )

    def generate(
        self,
        messages,
        stop_sequences=None,
        response_format=None,
        tools_to_call_from=None,
        **kwargs,
    ) -> ChatMessage:
        """Choose a tool or return final_answer."""

        # TODO:
        # - identify the original user question;
        # - identify the latest TOOL_RESPONSE observation;
        # - when an observation exists, call final_answer;
        # - for "add", call math_tool with op="add";
        # - for "multiply", call math_tool with op="multiply";
        # - otherwise call kb_lookup;
        # - preserve [kb:N] citations;
        # - admit when the observation says that no evidence was found.

        raise NotImplementedError

# 4. Optional local Transformers model

In [ ]:
# Keep False for the default, reliable stub run.
USE_LOCAL_TRANSFORMERS_MODEL = False

MODEL_ID = "sshleifer/tiny-gpt2"

if USE_LOCAL_TRANSFORMERS_MODEL:
    model = TransformersModel(
        model_id=MODEL_ID,

        # TODO: Choose reasonable local generation options.
        # device_map="auto",
        # max_new_tokens=...,
        # do_sample=False,
    )
else:
    model = DeterministicToolModel()

print("Model:", model.model_id)

# 5. Create the ToolCallingAgent

In [ ]:
# TODO:
# - add both custom tools;
# - set max_steps to at least 3;
# - keep final answers between 2 and 4 short sentences;
# - tell the agent to preserve [kb:N] citations;
# - tell it to admit missing evidence.

agent = ToolCallingAgent(
    tools=[
        # kb_tool,
        # math_tool,
    ],
    model=model,
    max_steps=3,
    verbosity_level=2,
    instructions=(
        "TODO: Add clear instructions for tool selection, "
        "citations, brevity, and missing evidence."
    ),
)

print(agent)

# 6. Run the required questions

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for question in tests:
    print("\n" + "=" * 80)
    print("Question:", question)

    # TODO: Run the agent with a clean memory for every question.
    # result = agent.run(...)

    # TODO: Print the final answer and inspect agent.memory.steps.

# 7. Missing-evidence check

In [ ]:
missing_question = (
    "What is the maintenance schedule for the Orion spacecraft?"
)

# TODO:
# Run the question and verify that the answer:
# - says the KB has insufficient evidence;
# - proposes a more specific follow-up question.

# Student checklist

- [ ] 5–8 source-tagged KB snippets
- [ ] `KBLookupTool`
- [ ] keyword matching
- [ ] clear no-evidence response
- [ ] `MathTool`
- [ ] add and multiply
- [ ] deterministic stub model
- [ ] optional `TransformersModel`
- [ ] `ToolCallingAgent`
- [ ] three required test questions
- [ ] tool-call inspection
- [ ] `[kb:1]` citation in the KB answer
- [ ] missing-evidence follow-up